# Water use comparison

In [ ]:
# basic
from pathlib import Path
import yaml
import numpy as np
import os

# spatial
import geopandas as gpd
import xarray as xr

# plotting
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from sklearn.metrics import mean_absolute_error
from scipy.stats import pearsonr
from sklearn.preprocessing import MinMaxScaler

path_data_raw = Path("/home/rooda/Datasets")
wd = Path('/home/rooda/OneDrive/Projects/Whatif-Andes')
os.chdir(wd)

# import config 
with open(wd / "config.yml") as f:
    cfg = yaml.safe_load(f)

from src.utils import extract_attributes

## Data

In [ ]:
basins_target_all = gpd.read_file("data/hydro/basins_hydroATLAS_lev08.gpkg")
#basins_target_all = basins_target_all[basins_target_all.glacier_area_RGI60 > 0] # only the modeled ones

In [ ]:
ds_global_irr  = xr.open_dataset(path_data_raw / "WATER_USE/Huang_HESS/Water_use_Huang.nc").cons_irr
ds_global_irr  = ds_global_irr.sel(date=slice("1990-01-01", "2010-12-31")) # overlappping period with runoff data
ds_global_irr  = ds_global_irr.resample(date="YS").sum() # annual sum
ds_global_irr  = ds_global_irr.mean(dim = "date")

sectors = ["DrinkingWater", "UnbilledDrinkingWater", "Mining", "Industry", "Livestock"] # remove energy for nonconsumtive use
ds_cr2wu_nlu = xr.open_dataset(path_data_raw / "WATER_USE/CR2WU/CR2WU_NLU_1960_2020.nc")
ds_cr2wu_nlu["Total_NLU"] = sum(ds_cr2wu_nlu[sector] for sector in sectors)
ds_cr2wu_nlu = ds_cr2wu_nlu["Total_NLU"]
ds_cr2wu_nlu = ds_cr2wu_nlu.sel(time = range(1990, 2011))
ds_cr2wu_nlu = ds_cr2wu_nlu.mean(dim = "time")

ds_cr2wu_lulucf = xr.open_dataset(path_data_raw / "WATER_USE/CR2WU/CR2WU_LULUCF_1960_2020.nc")
ds_cr2wu_lulucf = ds_cr2wu_lulucf.sel(time = range(1990, 2011))
ds_cr2wu_lulucf = ds_cr2wu_lulucf.mean(dim = "time")

ds_global  = xr.open_dataset(path_data_raw / "WATER_USE/Huang_HESS/Water_use_Huang.nc").cons_total
ds_global  = ds_global.sel(date=slice("1990-01-01", "2010-12-31")) # overlappping period with runoff data
ds_global  = ds_global.resample(date="YS").sum() # annual sum
ds_global  = ds_global.mean(dim = "date")
ds_global  = ds_global.interp(lon = ds_cr2wu_nlu.x, lat = ds_cr2wu_nlu.y)
ds_global  = ds_global.where(ds_cr2wu_nlu>0)

## Comparison at the basin scale

In [ ]:
basins_target_all = extract_attributes(ds_cr2wu_nlu, basins_target_all, "ds_cr2wu_nlu")
basins_target_all = extract_attributes(ds_cr2wu_lulucf.Irrigated_Consumptive, basins_target_all, "ds_cr2wu_lulucf_irrigated")
basins_target_all = extract_attributes(ds_cr2wu_lulucf.Rainfed_Consumptive, basins_target_all, "ds_cr2wu_lulucf_rainfed")
basins_target_all = extract_attributes(ds_cr2wu_lulucf.Total_Consumptive, basins_target_all, "ds_cr2wu_lulucf_total")
basins_target_all = extract_attributes(ds_global, basins_target_all, "ds_global")
basins_target_all = extract_attributes(ds_global_irr, basins_target_all, "ds_global_irr")

# Use where to mask values
basins_target_all["ds_global"] = basins_target_all["ds_global"].where(basins_target_all["ds_cr2wu_nlu"] > 0, np.nan)

# the true consumptive total (?)
basins_target_all["ds_cr2wu_total_consumptive"] = (basins_target_all["ds_cr2wu_lulucf_total"] + basins_target_all["ds_cr2wu_nlu"])

In [ ]:
# Performance metrics
valid = basins_target_all.dropna(subset=["ds_cr2wu_total_consumptive", "ds_global"])

r_pearson, _ = pearsonr(valid.ds_cr2wu_total_consumptive, valid.ds_global)
mae   = mean_absolute_error(valid.ds_cr2wu_total_consumptive, valid.ds_global)
pbias = 100 * (valid.ds_global.sum() - valid.ds_cr2wu_total_consumptive.sum()) \
              / valid.ds_cr2wu_total_consumptive.sum()

print(f"Pearson-r: {r_pearson:.2f}")
print(f"MAE: {mae:.2f}")
print(f"Percent Bias: {pbias:.2f}%")

## Plot

In [ ]:
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.08)
scaler = MinMaxScaler(feature_range=(7, 25))

# 1. Scatter plot
fig.add_trace(go.Scatter(x=[-30, 500], y=[-30, 500],
                         mode="lines", name="1:1 line",
                         line=dict(color="#BAC7DA", dash='dot'),
                         showlegend=False), row=1, col=1)


fig.add_trace(go.Scatter(x=valid["ds_global"], y=valid["ds_cr2wu_total_consumptive"],
                         mode="markers", name="Basins",
                         marker=dict(color=valid['ds_global_irr'], 
                                     colorscale="aggrnyl_r",
                                     line = dict(color="#40663C", width=0.2), cmax = 250, cmin = 0,
                                     colorbar=dict(title="Irrigation<br>(mm yr<sup>-1</sup>)<br>", thickness=18, tickwidth=0.2),
                                     size=scaler.fit_transform(valid[['basin_area']]).flatten(),
                                     opacity=1), showlegend=False), row=1, col=1)


fig.update_xaxes(title_text="Huang et al. (mm yr<sup>-1</sup>)", range = [-30, 500], dtick = 100,  row=1, col=1)
fig.update_yaxes(title_text="CR2WU (mm yr<sup>-1</sup>)", range = [-30, 500], dtick = 100, row=1, col=1)

# 2. Histogram of difference
fig.add_trace(go.Histogram(x=valid["ds_cr2wu_total_consumptive"] - valid["ds_global"], 
                           histnorm='percent',
                           nbinsx=50, marker_color="#326D4C", 
                           opacity=0.9, showlegend=False), row=1, col=2)

fig.update_xaxes(title_text="Difference (mm yr<sup>-1</sup>)", range = [-150, 150], row=1, col=2)
fig.update_yaxes(title_text="Percent (%)", row=1, col=2)


fig.add_annotation(text="a)", xref="x domain", yref="y domain", x=0.02, y=0.98,
                   showarrow=False, font=dict(size=16), row=1, col=1)
fig.add_annotation(text="b)", xref="x domain", yref="y domain", x=0.02, y=0.98,
                   showarrow=False, font=dict(size=16), row=1, col=2)

fig.add_annotation(text=f"r-Pearson = {r_pearson:.2f}<br>MAE = {mae:.1f}<br>PBIAS = {pbias:.1f}%",
                   xref="x", yref="y", x=410, y=20,
                   showarrow=False, align="left",
                   font=dict(size=14), row=1, col=1)

# Update layout
fig.update_xaxes(ticks="outside", griddash="dot", zeroline=False, title_standoff=2)
fig.update_yaxes(ticks="outside", griddash="dot", zeroline=False, title_standoff=2)
fig.update_layout(
    height=400, width=900, font=dict(size=15, family ="Arial"),
    template="plotly", plot_bgcolor="rgba(229,236,246,0.5)",
    margin=dict(t=10, b=50, l=50, r=10)
)

fig.write_image("figures/fig_water_use_comparison.svg", height=400, width=900)
fig.show()